# Generate benchmark answers with both generators

Answers every benchmark question once with Qwen3.5-9B and Gemma 4 12B (4-bit, llama.cpp, thinking off, fixed seed), one model per T4, from the frozen evidence. Use the pilot questions first (60 questions, 120 answers), then the full set after the pilot decisions. Re-running skips answers already written. Expect about 1 GPU hour for the pilot and 3-5 for the full set.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Point to the built questions (a private Kaggle Dataset made from data/benchmark-build/, never the public repository)

In [ ]:
QUESTIONS = '/kaggle/input/faithguard-benchmark/questions.jsonl'  # change to your input path

## Start llama.cpp: one server per T4 (downloads 12.7 GB of model files the first time)

In [ ]:
from pathlib import Path
from faithguard.generation import llama
SERVER = llama.prepare(os.path.join(WORK, 'llama_work'))   # pinned llama.cpp release; builds from source if needed
servers = {}
for i, name in enumerate(['qwen', 'gemma']):                # one model per T4
    servers[name] = llama.LlamaServer(SERVER, llama.model_path(name), port=8080 + i, gpu=str(i), log=f'{OUT}/server-{name}.log')
for name, s in servers.items():
    print(name, 'ready in', round(s.wait_ready()), 's')

## Generate with both models at once (resumable)

In [ ]:
import threading
from faithguard.benchmark import BenchmarkQuestion
from faithguard.records import read_jsonl
from faithguard.generation.answers import generate_answers
from faithguard.generation.models import MODELS
questions = list(read_jsonl(QUESTIONS, BenchmarkQuestion))
print(len(questions), 'questions')
def run(name):
    n = generate_answers(questions, servers[name].url, name, MODELS[name]['sampling'], Path(OUT) / f'answers-{name}.jsonl')
    print(name, n, 'new answers')
threads = [threading.Thread(target=run, args=(n,)) for n in servers]
for t in threads: t.start()
for t in threads: t.join()

## Stop the servers and summarise

In [ ]:
import json, statistics
for s in servers.values(): s.stop()
for name in servers:
    rows = [json.loads(l) for l in open(Path(OUT) / f'answers-{name}.jsonl', encoding='utf-8')]
    speed = [r['settings']['timings'].get('predicted_per_second', 0) for r in rows]
    print(name, len(rows), 'answers; thinking leaks:', sum(r['settings']['thinking_leak'] for r in rows),
          '; empty:', sum(not r['text'] for r in rows), '; median tokens/s:', round(statistics.median(speed), 1) if speed else '-')

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])